In [ ]:
import numpy as np
from scipy.spatial import Delaunay
import zarr
import pandas as pd
import seaborn as sns
from sklearn.neighbors import NearestNeighbors

from util import *
from plotting import *

In [ ]:
# seaborn just wont shut up
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:

def play_sample(
    series,
    sample_rate_hz: int = 48_000,
    play_offset: int = None,
    play_len: int = None,
    autoplay: bool = True,
):
    from IPython.display import Audio

    series = np.asarray(series)
    if play_offset is not None:
        series = series[play_offset : play_offset + play_len]
    return Audio(series.astype(np.float32), rate=sample_rate_hz, autoplay=autoplay)

def min_max_scale(a):
    diff = a.max() - a.min()
    return (a - a.min()) / diff

def sample_by_idx(idx: int):
    return capture_buffers_zarr.blocks[int(idx)]
    
def min_sample(stat: str):
    idx = int(df[df[stat] == df[stat].min()].index[0])    
    return sample_by_idx(idx)

def max_sample(stat: str):
    idx = int(df[df[stat] == df[stat].max()].index[0])    
    return sample_by_idx(idx)
    

In [ ]:
RUN = '001'

class Opts:
    cv_buffers_zarr = f"runs/{RUN}/cv_buffers.z"
    capture_buffers_zarr = f"runs/{RUN}/capture_buffers.z"
    cv_samples_npy = f"runs/{RUN}/cv_samples.npy"

opts = Opts()

### read audio and calculate stats

In [ ]:
capture_buffers_zarr = zarr.open(opts.capture_buffers_zarr, mode="r")
print('nchunks', capture_buffers_zarr.nchunks)

In [ ]:
records = []
for b in range(capture_buffers_zarr.nchunks):    
    sample = capture_buffers_zarr.blocks[b]
    ch0_of_sample = sample[:,0]    
    stats = calculate_audio_stats(ch0_of_sample, ignore_in_out=500)
    records.append(stats)

In [ ]:
df = pd.DataFrame(records)
df.describe()

## min/max RMS ( quiet / loud )

In [ ]:

eg_sample = min_sample('rms')
plot(eg_sample[20_000:25_000 ,0])

In [ ]:
play_sample(eg_sample[10_000:35_000 ,0])

In [ ]:
eg_sample = max_sample('rms')
plot(eg_sample[20_000:25_000 ,0])

In [ ]:
play_sample(eg_sample[10_000:35_000 ,0])

## min/max sc ( dark, bright )

In [ ]:
eg_sample = min_sample('spectral_centroid')
plot(eg_sample[20_000:25_000 ,0])

In [ ]:
play_sample(eg_sample[10_000:35_000 ,0])

In [ ]:
eg_sample = max_sample('spectral_centroid')
plot(eg_sample[20_000:25_000 ,0])

In [ ]:
play_sample(eg_sample[10_000:35_000 ,0])

## min/max thd ( clean, distorted )

In [ ]:
eg_sample = min_sample('total_harmonic_dist')
plot(eg_sample[20_000:25_000 ,0])

In [ ]:
play_sample(eg_sample[10_000:35_000 ,0])

In [ ]:
eg_sample = max_sample('total_harmonic_dist')
plot(eg_sample[20_000:25_000 ,0])

In [ ]:
play_sample(eg_sample[10_000:35_000 ,0])

## min/max flatness

In [ ]:
eg_sample = min_sample('flatness')
plot(eg_sample[20_000:25_000 ,0])

In [ ]:
eg_sample = max_sample('flatness')
plot(eg_sample[20_000:25_000 ,0])

## min/max odd/even

In [ ]:
eg_sample = min_sample('odd_even')
plot(eg_sample[20_000:25_000 ,0])

In [ ]:
eg_sample = max_sample('odd_even')
plot(eg_sample[20_000:25_000 ,0])

### build triangulation for cv

In [ ]:
cv_buffers_zarr = zarr.open(opts.cv_buffers_zarr, mode="r")
print('nchunks', cv_buffers_zarr.nchunks)

In [ ]:
cv_samples = np.load(opts.cv_samples_npy)

# recall A is the last column and we want to include it!

cv_samples.shape

In [ ]:
df_wide = pd.DataFrame(cv_samples[:,:4], columns=['a_cv', 'b_cv', 'morph', '-'])
df_long = df_wide.melt(var_name='axis', value_name='value')
g = sns.displot(
    data=df_long, 
    x='value', 
    col='axis',
    col_wrap=2,  # 2x2
    kde=True, 
    facet_kws=dict(sharex=False, sharey=False)
)

In [ ]:
tri = Delaunay(cv_samples)
num_simplex_vertices = tri.simplices.shape[-1]
tri.simplices.shape

In [ ]:
edges = set()
for simplex in tri.simplices:
    for i in range(num_simplex_vertices):
        for j in range(i + 1, num_simplex_vertices):
            edge = tuple(sorted((simplex[i], simplex[j])))
            edges.add(edge)
unique_edges = list(edges)
len(unique_edges)

In [ ]:
midpoints = []   
for edge in unique_edges:
    pt_i, pt_j = edge
    midpoint = (cv_samples[pt_i] + cv_samples[pt_j]) / 2.0
    midpoints.append(midpoint)
midpoints = np.array(midpoints)

nn = NearestNeighbors(radius=0.01).fit(cv_samples)
density_counts = nn.radius_neighbors(midpoints, return_distance=False)
densities = np.array([len(neighbors) for neighbors in density_counts], dtype=float)

density_weight = 2.0  # higher => pushed apart more
density_scores = (1.0 + density_weight * densities)

### calculate local gradients

In [ ]:
def calculate_local_grads(audio_stat):
    local_grads = []
    for edge in unique_edges:
        pi, pj = edge
        cv_i, cv_j = cv_samples[pi], cv_samples[pj]
        cv_distance = np.linalg.norm(cv_i - cv_j)
        #print('cv_i cv_j', cv_i, cv_j, 'dist', cv_distance)
        stat_i, stat_j = audio_stat[pi], audio_stat[pj]
        stat_distance = np.abs(stat_i - stat_j)
        #print('stat_i, stat_j', stat_i, stat_j, 'dist', stat_distance)
        local_grad = stat_distance / cv_distance
        local_grads.append((local_grad, stat_distance, cv_distance, pi, pj))        
    return pd.DataFrame(local_grads, columns='local_grad stat_distance cv_distance pi pj'.split(' '))

### largest / smaller gradient difference for RMS

In [ ]:
audio_stat = np.array(df['rms'])
audio_stat = min_max_scale(audio_stat)
local_grads = calculate_local_grads(audio_stat)
local_grads.head()  

In [ ]:
sns.displot(local_grads[['stat_distance', 'cv_distance']])

### highest grads 

In [ ]:
highest_grads = local_grads.sort_values('local_grad', ascending=False)
highest_grads.head()

In [ ]:
sample_i = sample_by_idx(highest_grads.iloc[0]['pi'])
sample_j = sample_by_idx(highest_grads.iloc[0]['pj'])

plot(np.stack([
    sample_i[20_000:30_000 ,0],
    sample_j[20_000:30_000 ,0]]).T)

### largest / smaller gradient difference for flatteness

In [ ]:
audio_stat = np.array(df['flatness'])
audio_stat = min_max_scale(audio_stat)
local_grads = calculate_local_grads(audio_stat)
local_grads.head()  

In [ ]:
sns.displot(local_grads[['stat_distance', 'cv_distance']])

In [ ]:
local_grads.sort_values('stat_distance', ascending=True).head()

In [ ]:
furthest_stat_distances = local_grads.sort_values('stat_distance', ascending=False)
furthest_stat_distances.head()

In [ ]:
sample_i = sample_by_idx(furthest_stat_distances.iloc[0]['pi'])
sample_j = sample_by_idx(furthest_stat_distances.iloc[0]['pj'])

plot(np.stack([
    sample_i[20_000:30_000 ,0],
    sample_j[20_000:30_000 ,0]]).T)

### lowest gradient => smallest difference

In [ ]:
lowest_grads = local_grads.sort_values('local_grad', ascending=True)
lowest_grads.head()

In [ ]:
sample_i = sample_by_idx(lowest_grads.iloc[0]['pi'])
sample_j = sample_by_idx(lowest_grads.iloc[0]['pj'])

plot(np.stack([
    sample_i[20_000:30_000 ,0],
    sample_j[20_000:30_000 ,0]]).T)

### highest gradient => largest difference

In [ ]:
highest_grads = local_grads.sort_values('local_grad', ascending=False)
highest_grads.head()

In [ ]:
sample_i = sample_by_idx(highest_grads.iloc[0]['pi'])
sample_j = sample_by_idx(highest_grads.iloc[0]['pj'])

plot(np.stack([
    sample_i[20_000:30_000 ,0],
    sample_j[20_000:30_000 ,0]]).T)


### min/max gradients for spectral centroid; dark vs bright

In [ ]:
audio_stat = np.array(df['spectral_centroid'])
audio_stat = min_max_scale(audio_stat)
local_grads = calculate_local_grads(audio_stat)
local_grads.head()

In [ ]:
sns.displot(local_grads[['stat_distance', 'cv_distance']])

In [ ]:
lowest_grads = local_grads.sort_values('local_grad', ascending=True)
lowest_grads.head()

In [ ]:
sample_i = sample_by_idx(lowest_grads.iloc[0]['pi'])
sample_j = sample_by_idx(lowest_grads.iloc[0]['pj'])

plot(np.stack([
    sample_i[20_000:30_000 ,0],
    sample_j[20_000:30_000 ,0]]).T)


In [ ]:
highest_grads = local_grads.sort_values('local_grad', ascending=False)
highest_grads.head()

In [ ]:
sample_i = sample_by_idx(highest_grads.iloc[0]['pi'])
sample_j = sample_by_idx(highest_grads.iloc[0]['pj'])

plot(np.stack([
    sample_i[20_000:30_000 ,0],
    sample_j[20_000:30_000 ,0]]).T)


In [ ]:
play_sample(sample_by_idx(highest_grads.iloc[0]['pi'])[:,0])

In [ ]:
play_sample(sample_by_idx(highest_grads.iloc[0]['pj'])[:,0])

### min/max gradients for total harmonic distortion; clean vs distorted

In [ ]:
audio_stat = np.array(df['total_harmonic_dist'])
audio_stat = min_max_scale(audio_stat)
local_grads = calculate_local_grads(audio_stat)
local_grads.head()

In [ ]:
sns.displot(local_grads[['stat_distance', 'cv_distance']])

In [ ]:
lowest_grads = local_grads.sort_values('local_grad', ascending=True)
lowest_grads.head()

In [ ]:
sample_i = sample_by_idx(lowest_grads.iloc[0]['pi'])
sample_j = sample_by_idx(lowest_grads.iloc[0]['pj'])

plot(np.stack([
    sample_i[20_000:30_000 ,0],
    sample_j[20_000:30_000 ,0]]).T)


In [ ]:
highest_grads = local_grads.sort_values('local_grad', ascending=False)
highest_grads.head()

In [ ]:
sample_i = sample_by_idx(highest_grads.iloc[0]['pi'])
sample_j = sample_by_idx(highest_grads.iloc[0]['pj'])

plot(np.stack([
    sample_i[20_000:30_000 ,0],
    sample_j[20_000:30_000 ,0]]).T)


In [ ]:
play_sample(sample_by_idx(int(highest_grads.iloc[0]['pi']))[:,0])

In [ ]:
play_sample(sample_by_idx(int(highest_grads.iloc[0]['pj']))[:,0])

### audio stats combined, including distance

In [ ]:
rms = min_max_scale(np.array(df['rms']))
flatness = min_max_scale(np.array(df['flatness']))
spectral_centroid = min_max_scale(np.array(df['spectral_centroid']))

# calculate all cv distances
cv_distances = []
for edge in unique_edges:
    pi, pj = edge
    cv_i, cv_j = cv_samples[pi], cv_samples[pj]
    cv_distance = np.linalg.norm(cv_i - cv_j)
    cv_distances.append(cv_distance)

# make a normalised version for score contrib
normed_cv_distances = min_max_scale(np.array(cv_distances))

#gradient_weight = 1.0
#distance_weight = 1.0

scores = []  # gradients and distances
for e, edge in enumerate(unique_edges):
    pi, pj = edge    

    rms_dist = np.abs(rms[pi] - rms[pj])
    #flatness_dist = np.abs(flatness[pi] - flatness[pj])
    spectral_centroid_dist = np.abs(spectral_centroid[pi] - spectral_centroid[pj])    
    stat_distance = rms_dist + spectral_centroid_dist    
    local_grad = stat_distance / cv_distances[e]
    score = local_grad + normed_cv_distances[e]    
    scores.append((score, local_grad, normed_cv_distances[e], rms_dist, spectral_centroid_dist , pi, pj))

just_s = [lg[0] for lg in scores]
min(just_s), max(just_s)

In [ ]:
edge_scores_df = pd.DataFrame(scores, columns=['score', 'local_grad', 'norm_dist', 'rms_dist', 'sc_dist', 'pi', 'pj'])
edge_scores_df.head()

In [ ]:
g = sns.displot(data=edge_scores_df[['norm_dist']], kde=True)
#g.set(yscale="log")
g

In [ ]:
edge_scores_df = edge_scores_df[edge_scores_df['norm_dist']>0.01]
edge_scores_df.head()

In [ ]:
lowest_score = edge_scores_df.sort_values('score', ascending=True)
lowest_score.head()

In [ ]:
sample_i = sample_by_idx(lowest_score.iloc[0]['pi'])
sample_j = sample_by_idx(lowest_score.iloc[0]['pj'])

plot(np.stack([
    sample_i[20_000:30_000 ,0],
    sample_j[20_000:30_000 ,0]]).T)


In [ ]:
highest_scores = edge_scores_df.sort_values('score', ascending=False)
highest_scores.head(20)

In [ ]:
sample_i = sample_by_idx(highest_scores.iloc[0]['pi'])
sample_j = sample_by_idx(highest_scores.iloc[0]['pj'])

plot(np.stack([
    sample_i[20_000:30_000 ,0],
    sample_j[20_000:30_000 ,0]]).T)


### generate candidates

In [ ]:
print("|edges|", len(highest_scores))

N = 256
topN_pi = list(highest_scores.head(N)['pi'])
topN_pj = list(highest_scores.head(N)['pj'])

candidate_cvs = []
for pi, pj in zip(topN_pi, topN_pj):
    cv_i, cv_j = cv_samples[pi], cv_samples[pj]
    candidate_cvs.append((cv_i + cv_j)/2)
candidate_cvs = np.stack(candidate_cvs)
np.save('candidate_cvs.npy', candidate_cvs)
candidate_cvs.shape